In [1]:
import sys
import os
import urllib3
from configparser import ConfigParser

# Add your local ThreatConnect SDK to path
sys.path.append(r"Z:\HTOC\Data_Analytics\threatconnect")
from ThreatConnect import ThreatConnect
from RequestObject import RequestObject
from Owners import Owners

# Add your project repo to path
project_root = r"C:\Users\jaskew\Documents\project_repository\scripts\Data Movement\ThrearConnect-api-pull"
if project_root not in sys.path:
    sys.path.append(project_root)

from utils.config_loader import load_config

# Load API config
config_path = os.path.join(project_root, "utils", "config.json")
try:
    api_secret_key, api_access_id, api_base_url, api_default_org = load_config(config_path)
    display(f"Loaded config from: {config_path}")
    display(f"Base URL: {api_base_url}")
    display(f"Access ID: {api_access_id}")
    display(f"Default Org: {api_default_org}")
except Exception as e:
    display(f"[ERROR] Failed to load configuration: {e}")
    sys.exit(1)

# Disable SSL verification warnings (use cautiously)
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
verify_ssl = False

# Initialize ThreatConnect session
try:
    tc = ThreatConnect(api_access_id, api_secret_key, api_default_org, api_base_url)
    display("ThreatConnect initialized.")
except Exception as e:
    display(f"[ERROR] Failed to initialize ThreatConnect: {e}")
    sys.exit(1)

# Define the owner (organization scope)
owner = 'HTOC Org'

# Create a request object to fetch indicators (or other data)
try:
    ro = RequestObject()
    ro.set_http_method('GET')
    ro.set_owner(owner)
    ro.set_owner_allowed(True)
    # ro.set_resource_pagination(True)  # Uncomment if needed
    display("RequestObject successfully created.")
except Exception as e:
    display(f"[ERROR] Failed to initialize RequestObject: {e}")
    sys.exit(1)




'Loaded config from: C:\\Users\\jaskew\\Documents\\project_repository\\scripts\\Data Movement\\ThrearConnect-api-pull\\utils\\config.json'

'Base URL: https://hvs.threatconnect.com/api'

'Access ID: 09783848890162390382'

'Default Org: HTOC Org'

'ThreatConnect initialized.'

'RequestObject successfully created.'

In [3]:
import pandas as pd
from datetime import datetime, timedelta
import pytz
import urllib.parse

# Configuration for ThreatConnect indicator query
QUERY_LOOKBACK_DAYS = 730  # days of lastObserved activity to include
INDICATOR_TYPE_NAMES = [
    "Address", "EmailAddress", "File", "Host", "URL", "ASN", "CIDR",
    "Email Subject", "Hashtag", "Mutex", "Registry Key", "User Agent",
]
OWNER_NAMES = [
    'HTOC Org',
    'CISA Federal Feed',
    'CMS_CTI',
    'Crowdstrike Falcon Intelligence',
    'DHS CISCP',
    'Intel471',
    'Mandiant Advantage Threat Intelligence',
    'VA_TIP Data',
]
RESULT_PAGE_SIZE = 500  # keep this smaller; same fields, just paged

# Setup
cutoff = pd.Timestamp.utcnow()
start_date = (datetime.now(pytz.UTC) - timedelta(days=QUERY_LOOKBACK_DAYS)).date()
start = f"{start_date}T00:00:00Z"

type_names = INDICATOR_TYPE_NAMES
type_name_condition = ", ".join([f'"{t}"' for t in type_names])

list_of_owners = OWNER_NAMES

# Build owner IN (...) clause
owner_condition = ", ".join([f'"{o}"' for o in list_of_owners])

tql_raw = (
    f'ownerName IN ({owner_condition}) AND '
    f'typeName IN ({type_name_condition}) AND '
    f'lastObserved >= "{start}"'
)

tql_encoded = urllib.parse.quote(tql_raw)

final_results = []

# Query indicators (paginate so you don't 502 with heavy fields)
# Create a NEW RequestObject WITHOUT owner restriction to query across all owners
ro_multi = RequestObject()
ro_multi.set_http_method('GET')

result_start = 0
result_limit = RESULT_PAGE_SIZE

while True:
    try:
        # NOTE: same fields list you requested (tags,observations,associatedGroups,falsePositives,threatAssess)
        # Only change here is removing the trailing comma after threatAssess which can break parsing.
        ro_multi.set_request_uri(
            f'/v3/indicators?tql={tql_encoded}'
            f'&fields=tags,observations,associatedGroups,falsePositives,threatAssess'
            f'&resultStart={result_start}&resultLimit={result_limit}'
        )

        response = tc.api_request(ro_multi)

        ct = response.headers.get('content-type', '')
        if not ct.startswith('application/json'):
            raise RuntimeError(f"Non-JSON response ({ct}): {response.content[:200]}")

        results = response.json()
        data_items = results.get('data', []) or []

        # stop when no more results
        if not data_items:
            break

        final_results.append(results)
        result_start += result_limit

    except Exception as e:
        display(f"Failed to query indicators (start={result_start}): {e}")
        break

# Normalize results
normalized_data = []
for result in final_results:
    data_items = result.get('data', [])
    if not data_items:
        display("No data returned in API response:", result)
    for item in data_items:
        if isinstance(item, dict) and 'summary' in item:
            normalized_data.append(item)

if normalized_data:
    observed_src = pd.json_normalize(normalized_data)
    observed_src['indicator'] = observed_src['summary'].astype(str).str.split().str[0].str.strip()
    observed_src['lastObserved'] = pd.to_datetime(observed_src['lastObserved'], utc=True, errors='coerce')
    observed_src = observed_src[observed_src['lastObserved'] >= pd.to_datetime(start, utc=True)]
    
    # Create a 'sources' column by aggregating ownerName values per indicator
    sources_per_indicator = (
        observed_src.groupby('indicator')['ownerName']
        .apply(lambda x: ', '.join(sorted(set(x))))
        .reset_index()
        .rename(columns={'ownerName': 'sources'})
    )

    # Merge sources back into observed_src
    observed_src = observed_src.merge(sources_per_indicator, on='indicator', how='left')
    # Filter to keep only records where ownerName is 'HTOC Org'
    observed_src = observed_src[observed_src['ownerName'] == 'HTOC Org'].copy()
else:
    display("No valid indicator data found.")
    observed_src = pd.DataFrame()

display(observed_src)

,id,dateAdded,ownerId,ownerName,webLink,type,lastModified,rating,confidence,threatAssessRating,...,whoisActive,associatedGroups.data,source,address,lastFalsePositive,sha256,md5,text,indicator,sources
0,6755399465229287,2025-07-28T17:15:16Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2026-03-16T15:12:17Z,3.0,71.0,1.50,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,198.235.24.129,"DHS CISCP, HTOC Org"
1,6755399487000216,2026-01-09T15:38:47Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2026-03-16T15:12:04Z,3.0,71.0,1.50,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,205.210.31.47,"DHS CISCP, HTOC Org"
2,5629499577081591,2025-11-17T16:08:15Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2026-03-16T15:11:51Z,3.0,65.0,2.00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,167.94.138.62,"DHS CISCP, HTOC Org"
3,5629499578170430,2025-11-26T17:40:57Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2026-03-16T15:11:37Z,3.0,66.0,1.75,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,206.168.34.51,HTOC Org
4,5629499578170433,2025-11-26T17:40:59Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2026-03-16T15:11:23Z,3.0,66.0,1.00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,167.94.138.122,"DHS CISCP, HTOC Org"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6553,5265344,2025-01-23T20:38:37Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,URL,2025-02-26T23:33:01Z,5.0,91.0,5.00,...,NaN,"[{'id': 546895, 'dateAdded': '2025-01-23T20:38...",NaN,NaN,NaN,NaN,NaN,https://cfa.org,https://cfa.org,HTOC Org
6555,4883040,2024-09-09T11:22:09Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,URL,2025-01-15T23:24:02Z,3.0,90.0,3.00,...,NaN,"[{'id': 455233, 'dateAdded': '2024-09-09T11:22...",ACD R&F,NaN,NaN,NaN,NaN,https://www.shorturl.at/,https://www.shorturl.at/,HTOC Org
6567,4181696,2022-04-13T13:22:17Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,URL,2024-03-30T23:23:34Z,4.0,74.0,2.00,...,NaN,"[{'id': 135099, 'dateAdded': '2022-04-13T13:22...",NaN,NaN,NaN,NaN,NaN,https://bitbucket.org/,https://bitbucket.org/,HTOC Org
6569,223609,2016-12-13T19:56:35Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,URL,2024-01-25T15:22:09Z,3.0,89.0,3.00,...,NaN,"[{'id': 39254, 'dateAdded': '2016-12-15T15:18:...",HHS TICAP FireEye NX,NaN,NaN,NaN,NaN,http://www.southhillsortho.com/,http://www.southhillsortho.com/,HTOC Org


In [ ]:
from pymongo import MongoClient

Indicators = observed_src
# Mongo connection
uri = "mongodb://localhost:27017/"

client = MongoClient(uri)

# Database and collection
db = client["Indicator_Data"]
collection = db["Indicators"]

# Convert dataframe to records
records = Indicators.to_dict("records")

# Insert into MongoDB
collection.insert_many(records)

print(f"Inserted {len(records)} records into MongoDB")

Inserted 3839 records into MongoDB
